# Week 22, Day 4: Linters and formatters

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w022d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ast

tree = ast.parse("import os\nimport sys\nprint(sys.argv)")
imported = [alias.name for node in ast.walk(tree) if isinstance(node, ast.Import) for alias in node.names]
used = {node.id for node in ast.walk(tree) if isinstance(node, ast.Name)}
print([name for name in imported if name not in used])

> **Key idea.** Follow PEP 8: 4-space indents, `snake_case` functions and variables, `CapWords` classes, `UPPER_CASE` constants. A formatter (Ruff or Black) fixes layout automatically; a linter (Ruff, Flake8, Pylint) finds likely bugs without running the code. Configure them in `pyproject.toml` and run them on every save.

In [ ]:
import ast

source = '''
def Total(prices, seen=[]):
    try:
        return sum(prices)
    except:
        return None
'''
tree = ast.parse(source)
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        print("function", node.name, "on line", node.lineno, "with arguments", [a.arg for a in node.args.args])
        for d in node.args.defaults:
            if isinstance(d, (ast.List, ast.Dict, ast.Set)):
                print("  mutable default on line", d.lineno)
    if isinstance(node, ast.ExceptHandler) and node.type is None:
        print("bare except on line", node.lineno)

In [ ]:
import ast
print(ast.dump(ast.parse("x = a + 1"), indent=2))

### Exercise: Naming styles

Write `style_of(name)` returning `"snake_case"` (lower case letters, digits and underscores, starting with a letter or underscore), `"CapWords"` (starts with a capital, letters and digits only, contains at least one lower-case letter), `"UPPER_CASE"` (capitals, digits and underscores, with at least one capital and no lower case), or `"other"`. Single-letter names: `"x"` is snake_case, `"X"` is UPPER_CASE.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert style_of(\"total_price\") == \"snake_case\" and style_of(\"_helper\") == \"snake_case\" and style_of(\"x\") == \"snake_case\"\n    assert style_of(\"BankAccount\") == \"CapWords\" and style_of(\"HTTPServer\") == \"CapWords\"\n    assert style_of(\"MAX_SIZE\") == \"UPPER_CASE\" and style_of(\"X\") == \"UPPER_CASE\" and style_of(\"PI2\") == \"UPPER_CASE\"\n    assert style_of(\"totalPrice\") == \"other\" and style_of(\"Bank_Account\") == \"other\" and style_of(\"2fast\") == \"other\"")

### Exercise: Unused imports

Write `unused_imports(source)` returning a sorted list of `(line, name)` for imports that are never used. Handle `import a`, `import a as b` (the name is `b`), `import a.b` (the name used is `a`), and `from a import b, c as d` (names `b` and `d`). A name counts as used if it appears as an `ast.Name` node anywhere.

In [ ]:
import ast

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    src = \"import os\\nimport sys as s\\nimport os.path\\nfrom math import pi, sqrt as root\\nfrom json import dumps\\nprint(s.argv, pi, os.sep)\\n\"\n    assert unused_imports(src) == [(4, \"root\"), (5, \"dumps\")]\n    assert unused_imports(\"print(1)\") == []\n    assert unused_imports(\"import re\\n\") == [(1, \"re\")]")

### Exercise: Layout checks

Write `layout_problems(source, max_length=79)` returning a list of `(line_number, code)` pairs, in line order (and in the order below within a line), for: `"W191"` a line indented with a tab; `"W291"` trailing spaces or tabs; `"E501"` a line longer than `max_length`; and `"W292"` (on the last line) if the file doesn't end with a newline. Lines are numbered from 1.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    src = \"x = 1\\n\\ty = 2   \\n\" + \"z = '\" + \"a\" * 80 + \"'\\nend\"\n    assert layout_problems(src) == [(2, \"W191\"), (2, \"W291\"), (3, \"E501\"), (4, \"W292\")]\n    assert layout_problems(\"ok = 1\\n\") == []\n    assert layout_problems(\"a = 1\\n\" * 3, max_length=4) == [(1, \"E501\"), (2, \"E501\"), (3, \"E501\")]")

### Exercise: Bug-shaped code

Write `danger_checks(source)` returning a sorted list of `(line, code)` for: `"E722"` a bare `except:` (an `ast.ExceptHandler` whose `type` is `None`); `"B006"` a function default that's a list, dictionary or set literal; and `"E711"` a comparison with `== None` or `!= None` (an `ast.Compare` whose operator is `ast.Eq` or `ast.NotEq` and whose right-hand side is the constant `None`).

In [ ]:
import ast

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    src = \"\"\"def add(item, items=[]):\n    items.append(item)\n    return items\n\ndef f(x, opts={}):\n    if x == None:\n        return 0\n    try:\n        return 1 / x\n    except:\n        return None\n\ndef g(y=None):\n    return y is None\n\"\"\"\n    assert danger_checks(src) == [(1, \"B006\"), (5, \"B006\"), (6, \"E711\"), (10, \"E722\")]\n    assert danger_checks(\"x = 1\") == []")

### Exercise: A tiny formatter

Write `tidy(source)` that: replaces each tab at the **start** of a line with 4 spaces; removes trailing spaces and tabs from every line; collapses runs of more than 2 blank lines into exactly 2; removes blank lines at the very end; and makes the file end with exactly one newline (an empty file stays empty). It must not change anything else.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    src = \"def f():\\n\\treturn 1   \\n\\n\\n\\n\\ndef g():\\n\\t\\tpass\\n\\n\\n\"\n    assert tidy(src) == \"def f():\\n    return 1\\n\\n\\ndef g():\\n        pass\\n\"\n    assert tidy(\"\") == \"\" and tidy(\"x = 1\") == \"x = 1\\n\"\n    assert tidy(\"s = 'a\\\\tb'\\n\") == \"s = 'a\\\\tb'\\n\"\n    assert tidy(tidy(src)) == tidy(src), \"formatting twice should change nothing\"")

### Bug hunt: What the linter said

A linter reported three problems with this code: `F841 local variable 'average' is assigned to but never used`, `E711 comparison to None should be 'is None'` and `E722 do not use bare except`. The first is a real bug: the function returns the wrong thing. Fix all three, catching only `ZeroDivisionError`, so it prints `70.0` and `None`.

In [ ]:
def mean(marks):
    if marks == None:
        return None
    try:
        average = sum(marks) / len(marks)
        return sum(marks)
    except:
        return None

print(mean([60, 70, 80]))
print(mean([]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"70.0\\nNone\")\n    assert \"== None\" not in source and \"except:\" not in source and \"ZeroDivisionError\" in source")

### Bug hunt: Hiding a built-in

This code names a variable `list` (and another `sum`), which hides Python's own `list` and `sum`, so the last line crashes with `TypeError: 'list' object is not callable`. Rename the two variables (to `items` and `total`) so it prints `6` and `['a', 'b']`.

In [ ]:
list = [1, 2, 3]
sum = 0
for x in list:
    sum += x
print(sum)
print(list("ab"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"6\\n['a', 'b']\")\n    import ast\n    assigned = {t.id for n in ast.walk(ast.parse(source)) if isinstance(n, ast.Assign) for t in n.targets if isinstance(t, ast.Name)}\n    assert \"list\" not in assigned and \"sum\" not in assigned")

### Puzzle: Tidy imports

Arrange these imports in PEP 8 order: standard library (alphabetically), then third-party, then your own. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # the PEP 8 style for class names (one word)
b = ___   # True or False: a formatter can change what your code does
c = ___   # the number of spaces per indent level in PEP 8
d = ___   # the file where Ruff's settings usually go

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"CapWords\" and b is False and c == 4 and d == \"pyproject.toml\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: your own linter

Combine checks into a small linter. Write `lint(source, max_length=79)` returning a list of strings like `"3: F401 'os' imported but unused"`, sorted by line number and then by code, from these checks:

- `F401 'NAME' imported but unused` (as in the unused-imports exercise);
- `E501 line too long (N > MAX)` with the actual length and the limit;
- `N802 function name 'NAME' should be lowercase` for a `def` whose name contains a capital letter;
- `F841 local variable 'NAME' is assigned to but never used` for a simple assignment `NAME = ...` inside a function whose name is never read (as an `ast.Name` with `ast.Load` context) anywhere in that function;
- `E722 do not use bare except`.

Then `summary(source)` returning a string such as `"3 problems: E501 x1, F401 x2"` (codes sorted, counts after `x`), or `"No problems"`.

In [ ]:
import ast

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SRC = \"\"\"import os\nimport sys\n\ndef CalculateTotal(prices):\n    result = 0\n    unused = len(prices)\n    for p in prices:\n        result += p\n    try:\n        return result\n    except:\n        return 0\n\nprint(CalculateTotal([1, 2]), sys.argv[0] if sys.argv else \"this line is deliberately written to be far too long ok\")\n\"\"\"\ndef test_lint():\n    \"lint\"\n    assert lint(SRC) == [\n        \"1: F401 'os' imported but unused\",\n        \"4: N802 function name 'CalculateTotal' should be lowercase\",\n        \"6: F841 local variable 'unused' is assigned to but never used\",\n        \"11: E722 do not use bare except\",\n        \"14: E501 line too long (117 > 79)\",\n    ]\n    assert lint(\"x = 1\\n\") == []\ndef test_summary():\n    \"summary\"\n    assert summary(SRC) == \"5 problems: E501 x1, E722 x1, F401 x1, F841 x1, N802 x1\"\n    assert summary(\"x = 1\\n\") == \"No problems\"")

## Warm-up

Four lines of Python found an unused import without running the code. That's what a linter does, with hundreds of checks like this one. By the end of today you'll have written several.

## PEP 8

**PEP 8** is Python's official style guide ("PEP" means Python Enhancement Proposal). Its main points:

**Layout**
- Indent with **4 spaces**, never tabs.
- Keep lines reasonably short: PEP 8 says 79 characters; many teams use 88 or 100.
- Two blank lines around top-level functions and classes; one between methods inside a class.
- Spaces around operators and after commas: `x = a + b`, `f(a, b)`; but none just inside brackets, and none around `=` in keyword arguments: `f(a, key=1)`.
- Imports at the top, one per line, grouped: standard library, third-party, your own.

**Naming**

| Kind | Style | Example |
|---|---|---|
| variables, functions, modules | `snake_case` | `total_price`, `read_file`, `bank_rules.py` |
| classes | `CapWords` | `BankAccount` |
| constants | `UPPER_SNAKE_CASE` | `MAX_SIZE` |
| "private" names | leading underscore | `_helper` |

Avoid single letters `l`, `O` and `I` (they look like 1 and 0), and names that hide built-ins (`list`, `str`, `sum`, `id`).

**Idioms**
- Compare with `None` using `is`: `if x is None`, not `if x == None`.
- Test emptiness directly: `if items:`, not `if len(items) > 0:`.
- Catch specific exceptions, never a bare `except:`.

The whole guide is at peps.python.org/pep-0008. It ends by saying consistency within a project matters most, and that a rule can be broken when following it makes code harder to read.

## Formatters

A **formatter** rewrites your file's layout to one consistent style: indentation, spacing, line breaks, quotes. It never changes what the code does. The two you'll meet:

- **Black**: "the uncompromising formatter", with almost no settings, which is the point.
- **Ruff**: a very fast tool that formats (in Black's style) and lints, in one program.

```
before:  def total( prices,tax = 0.2 ) :
            return sum( prices )*(1+tax)
after:   def total(prices, tax=0.2):
             return sum(prices) * (1 + tax)
```

The real gain isn't neatness. It's that the team never discusses layout again, and code review diffs only show real changes.

## Linters

A **linter** reads code without running it, looking for patterns that are usually mistakes or make code hard to maintain. Ruff, Flake8 and Pylint are the common ones. Each warning has a code and a message:

| Code (Ruff) | Message | Why it matters |
|---|---|---|
| F401 | `os` imported but unused | clutter, and a hint something was forgotten |
| F841 | local variable `result` assigned but never used | often a bug: you meant to use it |
| F821 | undefined name `totl` | a typo that will crash when that line runs |
| E722 | do not use bare `except` | it hides every error, even typos and Ctrl+C |
| B006 | mutable default argument | the shared-default bug from Week 7 |
| E711 | comparison to `None` should be `is None` | `==` can be fooled by odd objects |
| N802 | function name should be lowercase | consistency |
| E501 | line too long | readability |

Linters catch real bugs. F821 finds a misspelt name in a branch your tests never reached; B006 finds a bug that's very hard to see by reading.

## Setting them up

On your own computer, in your project's virtual environment:

```
pip install ruff
ruff format .          # format every file in the folder
ruff check .           # lint every file
ruff check --fix .     # fix what it safely can (such as removing unused imports)
```

Settings go in a file called `pyproject.toml` at the top of the project, the standard place for Python project settings:

```
[tool.ruff]
line-length = 88

[tool.ruff.lint]
select = ["E", "F", "B", "N", "I"]     # which families of checks to run
```

In **VS Code**, install the **Ruff** extension, then in Settings turn on **Format On Save** and choose Ruff as the default Python formatter. Now every save formats the file, and lint problems appear as squiggles as you type.

Teams also run the checks automatically before each commit with a tool called `pre-commit`, and on every pull request (Week 24 sets up automatic checks).

## Code as a tree

How does a linter "read" code? Python's `ast` module (**a**bstract **s**yntax **t**ree) turns source code into a tree of nodes: a `FunctionDef` node for each `def`, a `Call` node for each call, a `Name` node for each name used, and so on. `ast.walk(tree)` visits every node; `isinstance` checks what kind it is. Every node knows its line number (`node.lineno`).

`ast.dump(tree, indent=2)` prints the whole tree, which is the best way to find out what node a piece of code becomes:

## Practice

## Quiz

## Daily challenge

## Before you go

In your `atlas` project: `pip install ruff`, then run `ruff check .` and `ruff format .` on the code you've written so far, and read what it finds. Turn on Format On Save in VS Code. Tomorrow: reading documentation, the skill that lets you learn any library on your own.